In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

DATASET_ROOT = "/kaggle/input/datasets/naveedgull/tomato-leaf-disease/Tomato Leaf Disease"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

print("Train:", TRAIN_DIR)
print("Test :", TEST_DIR)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)
    if root.count(os.sep) >= 6:
        dirs[:] = []

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)

In [ ]:
class_names = train_ds.class_names
print("EXACT MODEL CLASS MAPPING:")
for i, name in enumerate(class_names):
    print(f"{i} -> {name}")
print("\nNumber of classes:", len(class_names))

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

In [ ]:
import os

base = 'data' # start from your data folder
print("Searching for folders with 10 subfolders...\n")

for root, dirs, files in os.walk(base):
    # PlantVillage usually has 10 class folders
    if len(dirs) >= 8: 
        print(f"Found: {root}")
        print(f"Subfolders: {sorted(dirs)}")
        print("-"*50)
    

In [ ]:
import tensorflow as tf

TRAIN_DIR = 'data/tomato-leaf-disease/Tomato Leaf Disease/train'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
print("\nEXACT MODEL CLASS MAPPING - DO NOT CHANGE THIS ORDER:")
for i, name in enumerate(class_names):
    print(f"{i} -> {name}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model('model/crop_disease_model.h5')

# If you used ImageDataGenerator or flow_from_directory
# This prints the exact class order the model was trained on
print(model.output_shape) # should be (None, 10)

# Check if you saved class_names in the notebook
# Common code: class_names = train_generator.class_indices
# If you have it, print: print(class_names)

In [ ]:
import tensorflow as tf
import json
import os

model = tf.keras.models.load_model('model/crop_disease_model.h5')
print("Model Output Shape:", model.output_shape)

# METHOD 1: If you used flow_from_directory and saved class_indices
try:
    with open('class_indices.json', 'r') as f:
        class_indices = json.load(f)
    # Flip dict: {0: 'class_name'}
    class_names = {v: k for k, v in class_indices.items()}
    class_names = [class_names[i] for i in range(len(class_names))]
    print("\nCLASS ORDER FROM class_indices.json:")
    for i, name in enumerate(class_names):
        print(f"{i}: {name}")
except:
    print("class_indices.json not found")

# METHOD 2: Get from data folder - this is 99% accurate
data_dir = 'data/'
if os.path.exists(data_dir):
    class_names = sorted(os.listdir(data_dir))
    print("\nCLASS ORDER FROM data/ FOLDER:")
    for i, name in enumerate(class_names):
        print(f"{i}: {name}")
else:
    print("data/ folder not found")

In [ ]:
from tensorflow.keras.applications import EfficientNetV2B0

base_model = EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = keras.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)

In [ ]:
model.save("/kaggle/working/tomato_disease_model.keras")
print("Model saved.")

In [ ]:
print("Model input shape:", model.input_shape)

expected_shape = (None, 256, 256, 3)

if model.input_shape == expected_shape:
    print("✅ Model input shape is correct: 256 × 256 × 3")
else:
    print(f"❌ WRONG input shape. Expected {expected_shape}, got {model.input_shape}")

In [3]:
import tensorflow as tf

model = tf.keras.models.load_model('model/crop_disease_model.h5')
model.save('model/crop_disease_model.keras')  # new .keras format
print("Saved as .keras")

Saved as .keras


In [2]:
import os
print("Current folder:", os.getcwd())
print("Files in model folder:", os.listdir('model') if os.path.exists('model') else "model folder not found")


Current folder: /workspaces/SIH_2026-Early-detection-and-management-of-crop-diseases-and-pest-infestations
Files in model folder: ['crop_disease_model.h5']


In [ ]:
model.save("crop_disease_model.h5")
print("✅ Model saved successfully")